# Iliad — Phase A (Ingestion & OCR) workbench

**Question this notebook answers:** *how clean is the extracted text for each Iliad witness, and where does extraction quality break down?*

Phase A turns raw source materials (PDF / TXT / TEI-XML) into clean text with provenance
markers and an `IngestionReport`. It is **fully deterministic** — OCR + text extraction, no
LLM calls — so no API keys are needed to run this notebook.

The Iliad is multi-witness, and each witness exercises a different ingestion path:

| Witness | `source_type` | locale | copyright | path exercised |
|---|---|---|---|---|
| Murray 1924 | `txt` | en | public-domain | plain-text baseline |
| Venetus A | `txt` | grc | public-domain | pre-transcribed `.md` (skips Greek OCR) |
| Gnedich 1829 | `scanned-pdf` | ru | public-domain | **Tesseract OCR — the Cyrillic edge case** |
| Heumann 2021 | `pdf` | en | **not-public-domain** | copyright gate (skipped) |

**Workbench rules honored here** (see `notebooks/CLAUDE.md`):
- Never write to `output/`. We reuse the real `_ingest_*` handlers from `sisyphus.phases.phase_a`
  but redirect their output to `notebooks/scratch/` instead of `run_ingest` (which would write to
  `workspace/` *and* `output/iliad/` and would also trip the witness-collision guard).
- No API calls; clear all cell outputs before committing.
- To actually persist a witness into the pipeline, use the CLI (`sisyphus ingest ...`), not this notebook.


## 1. Setup

In [ ]:
import io
import sys
from pathlib import Path

from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table


# Resolve repo root so this works whether Jupyter was launched from the repo
# root or from notebooks/phase-a/ — walk up until we find pyproject.toml.
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Could not locate repo root (no pyproject.toml above cwd)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Phase A needs no API keys; load .env only for parity with the AI phases.
load_dotenv(REPO_ROOT / ".env")

console = Console()

# Workbench rule: notebooks NEVER write to output/. Everything goes to scratch.
SCRATCH = REPO_ROOT / "notebooks" / "scratch" / "phase-a"
SCRATCH.mkdir(parents=True, exist_ok=True)

# Collected per-witness rows for the final summary (keyed so re-runs overwrite).
RESULTS: dict[str, dict] = {}


def show_table(title, rows):
    if not rows:
        console.print(f"[dim]{title}: (no rows)[/dim]")
        return
    cols = list(rows[0].keys())
    t = Table(title=title)
    for c in cols:
        t.add_column(str(c))
    for r in rows:
        t.add_row(*[str(r.get(c, "")) for c in cols])
    console.print(t)


def read_sample(out_dir: Path, n: int = 400) -> str:
    files = sorted(out_dir.glob("text-*.txt"))
    if not files:
        return "(no text output)"
    return files[0].read_text(encoding="utf-8")[:n]


print("repo root :", REPO_ROOT)
print("scratch   :", SCRATCH.relative_to(REPO_ROOT))


## 2. Survey the Iliad witnesses

Several Iliad manifests omit `source_file`, so we resolve the actual file explicitly. Gnedich uses
the canonical manifest under `sources/manifests/`, which carries both `source_file` and `ocr.lang`.

In [ ]:
from sisyphus.io.yaml_io import read_yaml

SOURCES = REPO_ROOT / "sources" / "iliad"
MANIFESTS = REPO_ROOT / "sources" / "manifests"

WITNESSES = [
    {"key": "murray-1924", "manifest": SOURCES / "manifest-murray-1924.yaml",
     "source_file": SOURCES / "iliad-murray.md"},
    {"key": "venetus-a", "manifest": SOURCES / "manifest-venetus-a.yaml",
     "source_file": SOURCES / "venetus-a-by-zone.md"},
    {"key": "gnedich-1829-ru", "manifest": MANIFESTS / "gnedich-1829-ru-iliad.yaml",
     "source_file": None},  # resolved from the manifest's source_file field
    {"key": "heumann-en", "manifest": SOURCES / "manifest-heumann-en.yaml",
     "source_file": SOURCES / "The-Iliad-Michael-Heumann.pdf"},
]

survey = []
for w in WITNESSES:
    m = read_yaml(w["manifest"])
    src = w["source_file"]
    if src is None:
        sf = m.get("source_file")
        src = (REPO_ROOT / sf) if sf else None
    w["resolved"] = src
    w["manifest_data"] = m
    survey.append({
        "key": w["key"],
        "source_type": m.get("source_type"),
        "locale": m.get("locale"),
        "ocr_lang": (m.get("ocr") or {}).get("lang", "—"),
        "copyright": m.get("copyright_status"),
        "file": src.name if src else "—",
        "exists": bool(src and src.exists()),
    })

show_table("Iliad witnesses", survey)


## 3. A workbench-safe ingest helper

We call the **real** Phase A handlers (`_ingest_txt`, `_ingest_digital_pdf`, `_ingest_tei_xml`) so the
notebook exercises production code — but we pass them a scratch output dir, never `output/`.
`scanned-pdf` is handled separately in §6 because its handler (`_ingest_scanned_pdf`) loops over the
*entire* document (438 pages for Gnedich) with no page limit.

In [ ]:
from sisyphus.phases.phase_a import (
    _ingest_digital_pdf,
    _ingest_tei_xml,
    _ingest_txt,
)
from sisyphus.schema import IngestionReport

# Source types run_ingest dispatches to (excluding scanned-pdf, handled in §6).
_HANDLERS = {
    "txt": _ingest_txt,
    "digital-pdf": _ingest_digital_pdf,
    "tei-xml": _ingest_tei_xml,
    "oracc-atf": _ingest_tei_xml,
}


def ingest_to_scratch(key, source_file, source_type):
    # Run the real handler for source_type, writing to scratch (not output/).
    # Returns (IngestionReport, out_dir).
    out = SCRATCH / key
    out.mkdir(parents=True, exist_ok=True)
    report = IngestionReport(
        run_id=f"nb-{key}", source_file=str(source_file), source_type=source_type
    )
    handler = _HANDLERS.get(source_type)
    if handler is None:
        report.errors.append(f"source_type '{source_type}' has no Phase A handler")
        return report, out
    handler(source_file, out, report, console)
    return report, out


## 4. Baseline — Murray 1924 (en, `txt`)

The simplest path: a clean public-domain plain-text witness. This is our quality baseline — if
extraction looks wrong here, the problem is upstream of any OCR concern.

In [ ]:
w = next(x for x in WITNESSES if x["key"] == "murray-1924")
rep, out = ingest_to_scratch("murray-1924", w["resolved"], w["manifest_data"]["source_type"])
console.print(rep.model_dump())
print("\n--- first 400 chars ---\n")
print(read_sample(out, 400))

RESULTS["murray-1924"] = {
    "witness": "murray-1924", "source_type": rep.source_type,
    "words": rep.word_count, "pages": rep.page_count,
    "ocr": rep.ocr_applied, "flagged": len(rep.flagged_pages),
    "note": "full ingest",
}


## 5. Greek — Venetus A (grc, `txt`)

Venetus A is ancient Greek, but its manifest deliberately points at the **pre-transcribed
`venetus-a-by-zone.md`** with `source_type: txt` — so it takes the plain-text path and *skips
Tesseract entirely*. This is by design: there is no `grc` Tesseract pack installed, and OCR of a
10th-century Greek minuscule manuscript would be far worse than the HMT transcription. Unicode Greek
flows through the text path untouched.

In [ ]:
w = next(x for x in WITNESSES if x["key"] == "venetus-a")
rep, out = ingest_to_scratch("venetus-a", w["resolved"], w["manifest_data"]["source_type"])
console.print(rep.model_dump())
print("\n--- first 500 chars ---\n")
print(read_sample(out, 500))

RESULTS["venetus-a"] = {
    "witness": "venetus-a", "source_type": rep.source_type,
    "words": rep.word_count, "pages": rep.page_count,
    "ocr": rep.ocr_applied, "flagged": len(rep.flagged_pages),
    "note": "pre-transcribed .md; no OCR",
}


## 6. OCR edge case — Gnedich 1829 (ru, `scanned-pdf`)

This is the heart of the notebook. The Gnedich Russian witness is a **scanned** PDF, so it must go
through Tesseract. If Tesseract runs in its default English mode against Cyrillic pages, it silently
substitutes the visually-nearest Latin glyph for each Cyrillic character — producing high-volume,
low-confidence garbage that segments cleanly and corrupts everything downstream
(see `doc/issue-ocr-cyrillic-language.md`).

Below we OCR the first few pages **twice** — once with `lang='eng'` (the broken default) and once
with `lang='rus'` (what the manifest's `ocr.lang` specifies) — and compare text + confidence
side by side. We loop inline over a handful of pages rather than calling `_ingest_scanned_pdf`,
which would OCR all 438 pages.

In [ ]:
import fitz
import pytesseract
from PIL import Image

w = next(x for x in WITNESSES if x["key"] == "gnedich-1829-ru")
gnedich_pdf = w["resolved"]
manifest_lang = (w["manifest_data"].get("ocr") or {}).get("lang", "eng")
print("manifest ocr.lang =", manifest_lang, "| source =", gnedich_pdf.name)

N_PAGES = 4  # title + first text pages; the full witness is 438 pages

try:
    langs = pytesseract.get_languages(config="")
except Exception:
    langs = []
has_rus = "rus" in langs
if not has_rus:
    print("⚠ Tesseract 'rus' pack not found — the fixed column will be blank.")
    print("  Install: brew install tesseract-lang (macOS) / apt-get install tesseract-ocr-rus")


def page_conf(img, lang):
    d = pytesseract.image_to_data(img, lang=lang, output_type=pytesseract.Output.DICT)
    cs = [c for c in d["conf"] if c != -1]
    return (sum(cs) / len(cs) / 100.0) if cs else 0.0


doc = fitz.open(str(gnedich_pdf))
ocr_rows = []
for i in range(min(N_PAGES, len(doc))):
    pix = doc[i].get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72))
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    eng = pytesseract.image_to_string(img, lang="eng").strip().replace("\n", " ")
    rus = (pytesseract.image_to_string(img, lang="rus").strip().replace("\n", " ")
           if has_rus else "")
    ocr_rows.append({
        "pg": i + 1,
        "conf_eng": round(page_conf(img, "eng"), 2),
        "conf_rus": round(page_conf(img, "rus"), 2) if has_rus else "—",
        "eng default (broken)": (eng[:38] or "(blank)"),
        "rus = manifest fix": (rus[:38] or "(blank)"),
    })
doc.close()

show_table("Gnedich OCR: eng default vs rus (manifest ocr.lang)", ocr_rows)

RESULTS["gnedich-1829-ru"] = {
    "witness": "gnedich-1829-ru", "source_type": "scanned-pdf",
    "words": "—", "pages": f"{min(N_PAGES, 438)}/438 probed",
    "ocr": True, "flagged": "see table",
    "note": "OCR probe; full run via CLI",
}


### What the table shows

- **`eng` (default):** Cyrillic mangled into Latin look-alikes (`ГОСУДАРСТВЕННОЕ` → `rocy JAPCTBEHHOE`),
  with confidence collapsing on text-heavy pages. The failure is *silent* — Tesseract never errors.
- **`rus` (manifest fix):** correct Cyrillic, confidence recovers sharply.

**Fix status — note a discrepancy.** `doc/issue-ocr-cyrillic-language.md` still says *"Unfixed"* and
references a function `_ocr_pdf` that does not exist in the code. But the current **working-tree**
`sisyphus/phases/phase_a.py` *already* threads `ocr_lang` from the manifest
(`run_ingest` reads `manifest["ocr"]["lang"]` → `_ingest_scanned_pdf(..., ocr_lang=...)`), and
`gnedich-1829-ru-iliad.yaml` already sets `ocr.lang: rus`. So the fix appears **implemented but
uncommitted**, and the doc describes the pre-fix state. Worth reconciling the doc with the code in a
follow-up (out of scope for this read-only notebook — don't edit the doc or commit code from here).

## 7. Copyright gate — Heumann 2021 (en, `pdf`)

The Heumann translation is **not public domain** (published 2021, author living). Phase A should not
process it into anything that could be exported. We surface the manifest flags and **skip**.

Two things to note: (1) its `source_type: pdf` is **not even a recognized Phase A type** —
`run_ingest` accepts `digital-pdf` / `scanned-pdf` / `tei-xml` / `oracc-atf` / `txt`, so this would
raise *"Unknown source_type"* anyway; (2) Murray 1924 is the EN public-domain witness to use instead.

In [ ]:
w = next(x for x in WITNESSES if x["key"] == "heumann-en")
m = w["manifest_data"]
print("copyright_status:", m.get("copyright_status"))
print("source_type     :", m.get("source_type"),
      "(NOT a recognized Phase A type)")

if m.get("copyright_status") != "public-domain":
    print("\n⛔ Skipping ingestion: Heumann 2021 is under copyright — do not process or export.")
    print("   Use murray-1924 as the EN public-domain witness.")
    note = "SKIPPED: copyrighted"
else:
    note = "would ingest"

RESULTS["heumann-en"] = {
    "witness": "heumann-en", "source_type": m.get("source_type"),
    "words": "—", "pages": "—", "ocr": "—", "flagged": "—", "note": note,
}


## 8. Summary

In [ ]:
order = ["murray-1924", "venetus-a", "gnedich-1829-ru", "heumann-en"]
summary = [RESULTS[k] for k in order if k in RESULTS]
show_table("Phase A — Iliad ingestion (scratch only; output/ untouched)", summary)


### Takeaways

1. **Plain-text witnesses (Murray, Venetus A) are clean** — the `txt` path is robust; Greek Unicode
   passes through untouched, which is exactly why Venetus uses a pre-transcribed `.md`.
2. **`scanned-pdf` is the risk surface.** OCR language must match the source script; the `eng` default
   silently corrupts Cyrillic. The fix is manifest-driven `ocr.lang` (present in the working tree).
3. **Copyright is gated at the manifest**, before any processing.

**To persist a witness into the real pipeline** (writes to `workspace/` + `output/`), use the CLI —
not this notebook:

```bash
sisyphus ingest sources/iliad/iliada-gnedich.pdf \
  --manifest sources/manifests/gnedich-1829-ru-iliad.yaml
```
